# 07 - Wan2.1: Text-to-Video Inference

Vizuara Inference Engineering Bootcamp - Multimodal Inference.

Video generation is image generation with one extra axis. We skip the
from-scratch build (a real video DiT does not fit on a single GPU) and
jump to a production model: **Wan2.1** (Alibaba, 2025). 14B parameters,
text-to-video, 5-second 720p clips, open weights.

There are two ways to run Wan2.1:

1. **Local pipeline** with the diffusers integration. Needs ~40 GB VRAM
   (A100 80 GB recommended).
2. **Hosted endpoint** via Replicate or fal. Pay per second, no GPU needed.

On Colab the local route is impractical. We demonstrate the call site
with both backends. Run the hosted backend if you do not have an A100.

**Runtime:** A100 for local (~5 minutes per clip), API call for hosted
(~30 seconds + queue).

## Backend A: Hosted via Replicate

Replicate hosts Wan2.1 behind a stable API. Get a token at
https://replicate.com/account/api-tokens and put it in `REPLICATE_API_TOKEN`.

In [ ]:
!pip -q install replicate requests pillow

In [ ]:
import os, requests
from getpass import getpass
from IPython.display import Video, display

if "REPLICATE_API_TOKEN" not in os.environ:
    os.environ["REPLICATE_API_TOKEN"] = getpass("Replicate API token: ")

import replicate

output = replicate.run(
    "wavespeedai/wan-2.1-t2v-480p",
    input={
        "prompt": "a paper boat floating down a slow river, hand-held camera, dappled sunlight, soft realistic look",
        "aspect_ratio": "16:9",
        "num_frames": 81,   # ~5 seconds at 16 fps
        "fps": 16,
    },
)
# Replicate returns either a URL or a FileOutput
url = output[0] if isinstance(output, list) else (output.url if hasattr(output, "url") else str(output))
print("video url:", url)

Replicate API token: ··········


ReplicateError: ReplicateError Details:
title: Insufficient credit
status: 402
detail: You have insufficient credit to run this model. Go to https://replicate.com/account/billing#billing to purchase credit. Once you purchase credit, please wait a few minutes before trying again.

## View the Clip

In [ ]:
# Download and display inline
video_path = "wan_output.mp4"
with open(video_path, "wb") as f:
    f.write(requests.get(url, timeout=120).content)
display(Video(video_path, embed=True, width=512))

## Backend B: Local With diffusers (A100, optional)

If you have a beefy GPU, you can run Wan2.1 locally. The code path is
almost identical to Flux except the pipeline is `WanPipeline` and the
output is a list of frames you encode to mp4. Memory is the dominant
constraint (the VAE alone is several GB).

In [ ]:
# !pip -q install diffusers transformers accelerate imageio[ffmpeg]
# from diffusers import WanPipeline
# import torch, imageio, numpy as np
#
# pipe = WanPipeline.from_pretrained(
#     "Wan-AI/Wan2.1-T2V-14B-Diffusers",
#     torch_dtype=torch.bfloat16,
# ).to("cuda")
# frames = pipe(
#     prompt="a paper boat floating down a slow river",
#     num_frames=81,
#     height=480, width=832,
#     num_inference_steps=30,
#     guidance_scale=5.0,
# ).frames[0]
# imageio.mimsave("wan_local.mp4", [np.array(f) for f in frames], fps=16)

## What Happens Inside

Even though we did not build this from scratch, the architecture follows
everything we have built. End-to-end:

1. **Text encoder** (T5 / CLIP) -> text token embeddings.
2. **3D VAE encoder** compresses the target video latents from
   `(81, 3, 480, 832)` pixels down to a much smaller `(T', C', H', W')`
   latent grid.
3. **Spatiotemporal DiT** denoises the latent grid, alternating
   spatial attention (within each frame) and temporal attention
   (across frames at each spatial location). Cross-attention injects the
   prompt at every block.
4. **3D VAE decoder** maps the final latent grid back to pixels.

The DDPM equations from notebook 05 still hold; the only thing that
changed is the dimensionality and the size of the network.

## Latency and Compute Reality

On an A100 80 GB, a 5-second 480p clip from Wan2.1-T2V takes ~3 to 5
minutes. The hosted backend is faster because Replicate runs an optimized
fork on H100s.

For comparison:
- Flux.1-schnell: 1 second per 768x512 image on A100.
- Wan2.1 T2V: 200+ seconds per 5-second 480p clip on A100.

A clip is roughly 100x slower to generate than a single image, and uses
far more memory. This is why video gen is batch, not interactive, as of
2026.

## Takeaways

- Wan2.1 is Stable Diffusion + a time axis. Same recipe, same equations.
- The bottleneck is compute and memory, not architecture: video is 100x
  more expensive than image because the latent grid is 100x bigger.
- For interactive workflows in 2026: image-to-video models (anchor on a
  still) are usually a better fit than text-to-video.
- The hosted-endpoint pattern shown here is the practical way to deliver
  video gen in a product: you decouple your service from a single H100.

Next up: notebook 08, RVQ audio tokenization from scratch.